# Preprocess datasets

This is an example code for loading datasets, visualizing dataset, calculating PCA basis (if necessary), and saving data in a unified format.
Procedures are as below:
1. ```load_[example_specific]_dataset```
    * **input**: example specific parameters
    * **output**: full_matrix, time_label
2. ```visualize_data```
    * **input**: full_matrix, time_label, colors
    * **output**: (scatter plots in 2D principal axes)
3. ```reduce_dimension```
    * **input**: full_matrix, d_reds, example_name
    * **output**: pca, projected_matrix, (pca variance ratio plot)
4. ```save_files```
    * **input**: data={time_label, full_matrix, projected_matrix, pca}, example_name
    * **output**: (pickle file for time_label, full_matrix, projected_matrix, pca stored in ```data``` directory)
      
It is written to be easily adapted to a new dataset from following two examples:

* EMT dataset
* Synthetic dataset

For tips to adapt to a new dataset, write an example-specific ```load_[example_specific]_dataset``` function. Datasets are typically matrices of gene expression samples obtained at different timepoints.
After loading the file, apply the universal functions in 2-4 in a row. 

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn import decomposition
import pickle
from IPython.display import Image, display

main_dir = os.getcwd()
if "notebooks" in main_dir:
    main_dir = main_dir[:-10]
    os.chdir(main_dir)
elif main_dir == "/content":
    main_dir = main_dir + "/GPA-source_code"
print(main_dir)
    
sys.path.append(main_dir)
data_dir = main_dir + '/data/'

contrast_colors = {
    0: '#1f77b4',  # blue
    1: '#2ca02c',  # green
    2: '#ff7f0e',  # orange
    3: '#8c564b',  # brown
    4: '#d62728',  # red 
    8: '#9467bd'  # purple (to be used for index 8)
}

In [ ]:
# universal functions
def visualize_data(full_matrix, time_label, colors={}):
    # full_matrix: numpy 2D array of size (sample_size x dimension)
    # time_label: time label for full_matrix
    # colors: dictionary of designated color labels (in [0,1]^3)

    # apply pca and project to 2D
    pca = decomposition.PCA(n_components=2, random_state=0)
    pca.fit(full_matrix)
    projected_matrix_2D = pca.transform(full_matrix)
    

    time_points = sorted(set(time_label))
    # random sample color
    if colors == {}:
        np.random.seed(127)
        for day in time_points:
            colors[day] = np.random.random(3)
    

    for day in time_points:
        fig = plt.figure(figsize=(4,3))

        plt.scatter(projected_matrix_2D[:,0], projected_matrix_2D[:,1], color='lightgray', alpha=0.3, s=0.7) # background
        vis_day = projected_matrix_2D[time_label==day,:]
        plt.scatter(vis_day[:,0], vis_day[:,1], alpha=1.0, color=colors[day], s=0.7, label = f'day{day}') # highlighted
        plt.legend()
        plt.show()    


def reduce_dimension(full_matrix, d_reds, example_name):
    # full_matrix: numpy 2D array of size (sample_size x dimension)
    # d_reds: list of reduced dimensions(int) for PCA analysis (strictly smaller than dimension)
    # example_name: specific name of the dataset (ex: "emt", "synthetic")
    if d_reds == []:
        print("No dimension reduction")
        return 
        
    dimension = full_matrix.shape[1]

    pca = decomposition.PCA(n_components=dimension, random_state=0)
    pca.fit(full_matrix)
    projected_matrix = pca.transform(full_matrix)

    # plot variance ratio
    variances = pca.explained_variance_ratio_ * 100
    
    plt.plot(range(1, len(variances)+1), np.cumsum(variances), '-')
    plt.xlabel("number of PCA basis", fontsize=16)
    plt.ylabel("Explained variance (%)", fontsize=16)
    
    for d_red in d_reds:
        plt.vlines(x=d_red, ymin=variances[0], ymax=np.sum(variances), color = 'r', linestyles='dashed')
        plt.text(x=d_red-0.01*dimension, y= np.sum(variances[:d_red])-3, s="$d'=%d$\n%.2f %%" % (d_red, np.sum(variances[:d_red])), fontsize=12)

    plt.ylim([variances[0], np.sum(variances)])
    plt.tick_params(axis='x', labelsize=14)
    plt.tick_params(axis='y', labelsize=14)
      
    plt.savefig(data_dir + example_name + "_pca_variance_ratios.png")
    plt.show() 

    return pca, projected_matrix


def save_files(data, example_name):
    time_label = data['time_label']
    full_matrix = data['full_matrix']
    try:
        projected_matrix = data['projected_matrix']
        pca = data['pca']
        pickle.dump([time_label, full_matrix, projected_matrix, pca], open(data_dir + example_name + "_preprocessed.pkl","wb"))
    except:
        pickle.dump([time_label, full_matrix], open(data_dir + example_name + "_preprocessed.pkl","wb"))

## EMT dataset

### Data property

* 6 snapshots at different timepoints (day 0, 1, 2, 3, 4, 8)
* samplesize: differ by days
* dimension: 72

### Data structure
Datasets are located in ```data/EMT_72genes``` directory. 


* ```submatrix_72_genes_expression_matrix.txt``` : Raw data matrix in $\mathbb{R}^{M \times N}$ is composed of $M$ rows for individual samples and $N$ columns of different genes.
* ```EMT_Cells_Days_matrix.txt``` : The entire dataset is divided into 6 classes which correspond to different days.


In [ ]:
def load_emt_dataset():
    import pandas as pd

    # gene_expression dataset => full_matrix
    GE_matrix_file = data_dir + 'EMT_72genes/submatrix_72_genes_expression_matrix.txt'
    if os.path.exists(GE_matrix_file):
        df = pd.read_table(GE_matrix_file, sep="\t")
        print(df)
        full_matrix = np.array(df)[:,1:]
    else:
        print("If full gene expression matrix is stored in the directory, load and view it.")

    # time information => time_label
    time_info_file = data_dir + 'EMT_72genes/EMT_Cells_Days_matrix.txt'
    df_cls = pd.read_table(time_info_file, sep="\t")
    time_label = np.array(df_cls['day'])
    print("classes:", set(time_label))
        
    return full_matrix, time_label

In [ ]:
example_name = 'emt_72' # 
d_reds = [2, 4, 8, 16, 32, 64, 72] # [] or list of reduced dimensions
    
full_matrix, time_label = load_emt_dataset()
visualize_data(full_matrix, time_label, colors=contrast_colors)
try:
    pca, projected_matrix = reduce_dimension(full_matrix, d_reds, example_name)
    data = {'full_matrix': full_matrix, 'time_label': time_label, 'pca': pca, 'projected_matrix': projected_matrix}
except:
    data = {'full_matrix': full_matrix, 'time_label': time_label}
save_files(data, example_name)

## Synthetic dataset
### Data property

* 5 snapshots at different timepoints (day 0 to day 4) 
* samplesize : 239 
* dimenison : 26

### Data structure
Datasets are located in ```data/traj_dataset``` directory as ```log_traj_#.log``` for different days.



In [ ]:
def load_synthetic_dataset():
    import pandas as pd

    # gene_expression dataset & time information
    cls = [0, 1, 2, 3, 4]
    mats = {}
    for c in cls:
        GE_matrix_file = data_dir + 'traj_dataset/log_traj_%d.log' % c

        if os.path.exists(GE_matrix_file):
            df = pd.read_table(GE_matrix_file, sep="\t", header=None)
            #print(df)
            mats[c] = np.array(df)[:,:-1]
        else:
            print("Missing data at day %d." % c)

    print("Dimension of selected genes = ", mats[0].shape[1])

    full_matrix = np.concatenate([mats[c] for c in cls])
    time_label = [c*np.ones(mats[c].shape[0], dtype=np.int32) for c in cls]
    time_label = np.concatenate(time_label)

    return full_matrix, time_label

In [ ]:
example_name = 'synthetic' # 
d_reds = [2, 4, 8, 16, 26] # [] or list of reduced dimensions
    
full_matrix, time_label = load_synthetic_dataset()
visualize_data(full_matrix, time_label, colors=contrast_colors)
try:
    pca, projected_matrix = reduce_dimension(full_matrix, d_reds, example_name)
    data = {'full_matrix': full_matrix, 'time_label': time_label, 'pca': pca, 'projected_matrix': projected_matrix}
except:
    data = {'full_matrix': full_matrix, 'time_label': time_label}
save_files(data, example_name)

## Stem Cell Differentiation dataset

### Data property

* 5 snapshots at different timepoints (day 0, 1, 2, 3, 4)
* samplesize: differ by days
* dimension: 101

### Data structure
Datasets are located in ```data/stem_Cell_Differentiation``` directory. 


* ```Data_0_gene_expression_matrix.txt``` : Raw data matrix in $\mathbb{R}^{M \times N}$ is composed of $M$ rows for individual samples and $N$ columns of different genes.
* ```Data_0_Cells_Days_matrix.txt``` : The entire dataset is divided into 6 classes which correspond to different days.


In [ ]:
def load_stem_cell_diff_dataset():
    import pandas as pd

    # gene_expression dataset => full_matrix
    GE_matrix_file = data_dir + 'Stem_Cell_Differentiation/Data_0_gene_expression_matrix.txt'
    if os.path.exists(GE_matrix_file):
        df = pd.read_table(GE_matrix_file, sep="\t")
        print(df)
        full_matrix = np.array(df)[:,1:]
    else:
        print("If full gene expression matrix is stored in the directory, load and view it.")

    # time information => time_label
    time_info_file = data_dir + 'Stem_Cell_Differentiation/Data_0_Cells_Days_matrix.txt'
    df_cls = pd.read_table(time_info_file, sep="\t")
    time_label = np.array(df_cls['day'])
    print("classes:", set(time_label))
        
    return full_matrix, time_label

In [ ]:
example_name = 'stem_cell_differentiation' # 
d_reds = [2, 4, 8, 16,32, 64,101] # [] or list of reduced dimensions
    
full_matrix, time_label = load_stem_cell_diff_dataset()
visualize_data(full_matrix, time_label, colors=contrast_colors)
try:
    pca, projected_matrix = reduce_dimension(full_matrix, d_reds, example_name)
    data = {'full_matrix': full_matrix, 'time_label': time_label, 'pca': pca, 'projected_matrix': projected_matrix}
except:
    data = {'full_matrix': full_matrix, 'time_label': time_label}
save_files(data, example_name)

## MCF7 cell line dataset

### Data property

* 2 snapshots at different timepoints 
* samplesize: differ by days
* dimension: 118


### Data structure
Datasets are located in ```data/MCF7 Cell Line`` directory. 


* ```combined_matrix_transposed_basal_NDPR_Rgene.txt``` : Raw data matrix in $\mathbb{R}^{M \times N}$ is composed of $M$ rows for individual samples and $N$ columns of different genes.
* ```Cells_Days_basal_NDPR.txt``` : The entire dataset is divided into 2 classes which correspond to different days.

In [ ]:
def load_MCF7_dataset():
    import pandas as pd

    # gene_expression dataset => full_matrix
    GE_matrix_file = data_dir + 'MCF7 Cell Line/combined_matrix_transposed_basal_NDPR_Rgene.txt'
    if os.path.exists(GE_matrix_file):
        df = pd.read_table(GE_matrix_file, sep="\t")
        print(df)
        full_matrix = np.array(df)[:,1:]
    else:
        print("If full gene expression matrix is stored in the directory, load and view it.")

    # time information => time_label
    time_info_file = data_dir + 'MCF7 Cell Line/Cells_Days_basal_NDPR.txt'
    df_cls = pd.read_table(time_info_file, sep="\t")
    time_label = np.array(df_cls['day'])
    print("classes:", set(time_label))
        
    return full_matrix, time_label

In [ ]:
example_name = 'MCF7' # 
d_reds = [2, 4, 8, 16, 32, 64, 72] # [] or list of reduced dimensions
    
full_matrix, time_label = load_MCF7_dataset()
visualize_data(full_matrix, time_label, colors=contrast_colors)
try:
    pca, projected_matrix = reduce_dimension(full_matrix, d_reds, example_name)
    data = {'full_matrix': full_matrix, 'time_label': time_label, 'pca': pca, 'projected_matrix': projected_matrix}
except:
    data = {'full_matrix': full_matrix, 'time_label': time_label}
save_files(data, example_name)

## Patient PA3 dataset

### Data property

* 2 snapshots at different timepoints 
* samplesize: differ by days
* dimension: 117


### Data structure
Datasets are located in ```data/Patient_PA3`` directory. 


* ```combined_matrix_transposed_palbo_BMC_nofibroblast_malignant_Rgene.txt``` : Raw data matrix in $\mathbb{R}^{M \times N}$ is composed of $M$ rows for individual samples and $N$ columns of different genes.
* ```Cells_Days_palbo_BMC_nofibroblast_malignant.txt``` : The entire dataset is divided into 2 classes which correspond to different days.

In [ ]:
def load_PA3_dataset():
    import pandas as pd

    # gene_expression dataset => full_matrix
    GE_matrix_file = data_dir + 'Patient_PA3/combined_matrix_transposed_palbo_BMC_nofibroblast_malignant_Rgene.txt'
    if os.path.exists(GE_matrix_file):
        df = pd.read_table(GE_matrix_file, sep="\t")
        print(df)
        full_matrix = np.array(df)[:,1:]
    else:
        print("If full gene expression matrix is stored in the directory, load and view it.")

    # time information => time_label
    time_info_file = data_dir + 'Patient_PA3/Cells_Days_palbo_BMC_nofibroblast_malignant.txt'
    df_cls = pd.read_table(time_info_file, sep="\t")
    time_label = np.array(df_cls['day'])
    print("classes:", set(time_label))
        
    return full_matrix, time_label

In [ ]:
example_name = 'PA3' # 
d_reds = [2, 4, 8, 16, 32, 64, 72] # [] or list of reduced dimensions
    
full_matrix, time_label = load_PA3_dataset()
visualize_data(full_matrix, time_label, colors=contrast_colors)
try:
    pca, projected_matrix = reduce_dimension(full_matrix, d_reds, example_name)
    data = {'full_matrix': full_matrix, 'time_label': time_label, 'pca': pca, 'projected_matrix': projected_matrix}
except:
    data = {'full_matrix': full_matrix, 'time_label': time_label}
save_files(data, example_name)

## Patient 862 dataset

### Data property

* 2 snapshots at different timepoints 
* samplesize: differ by days
* dimension: 116


### Data structure
Datasets are located in ```data/Patient_862`` directory. 


* ```combined_matrix_transposed_palbo_NatMed_862_nofibroblast_malignant_Rgene.txt``` : Raw data matrix in $\mathbb{R}^{M \times N}$ is composed of $M$ rows for individual samples and $N$ columns of different genes.
* ```cell_matrix_palbo_862_1.txt``` : The entire dataset is divided into 2 classes which correspond to different days.

In [ ]:
def load_862_dataset():
    import pandas as pd

    # gene_expression dataset => full_matrix
    GE_matrix_file = data_dir + 'Patient_862/combined_matrix_transposed_palbo_NatMed_862_nofibroblast_malignant_Rgene.txt'
    if os.path.exists(GE_matrix_file):
        df = pd.read_table(GE_matrix_file, sep="\t")
        print(df)
        full_matrix = np.array(df)[:,1:]
    else:
        print("If full gene expression matrix is stored in the directory, load and view it.")

    # time information => time_label
    time_info_file = data_dir + 'Patient_862/cell_matrix_palbo_862_1.txt'
    df_cls = pd.read_table(time_info_file, sep="\t")
    time_label = np.array(df_cls['day'])
    print("classes:", set(time_label))
        
    return full_matrix, time_label

In [ ]:
example_name = '862' # 
d_reds = [2, 4, 8, 16, 32, 64, 72] # [] or list of reduced dimensions
    
full_matrix, time_label = load_862_dataset()
visualize_data(full_matrix, time_label, colors=contrast_colors)
try:
    pca, projected_matrix = reduce_dimension(full_matrix, d_reds, example_name)
    data = {'full_matrix': full_matrix, 'time_label': time_label, 'pca': pca, 'projected_matrix': projected_matrix}
except:
    data = {'full_matrix': full_matrix, 'time_label': time_label}
save_files(data, example_name)

## Patient 887 dataset

### Data property

* 2 snapshots at different timepoints 
* samplesize: differ by days
* dimension: 116


### Data structure
Datasets are located in ```data/Patient_887`` directory. 


* ```combined_matrix_transposed_palbo_NatMed_887_nofibroblast_malignant_Rgene.txt``` : Raw data matrix in $\mathbb{R}^{M \times N}$ is composed of $M$ rows for individual samples and $N$ columns of different genes.
* ```cell_matrix_palbo_887_1.txt``` : The entire dataset is divided into 2 classes which correspond to different days.

In [ ]:
def load_887_dataset():
    import pandas as pd

    # gene_expression dataset => full_matrix
    GE_matrix_file = data_dir + 'Patient_887/combined_matrix_transposed_palbo_NatMed_887_nofibroblast_malignant_Rgene.txt'
    if os.path.exists(GE_matrix_file):
        df = pd.read_table(GE_matrix_file, sep="\t")
        print(df)
        full_matrix = np.array(df)[:,1:]
    else:
        print("If full gene expression matrix is stored in the directory, load and view it.")

    # time information => time_label
    time_info_file = data_dir + 'Patient_887/cell_matrix_palbo_887_1.txt'
    df_cls = pd.read_table(time_info_file, sep="\t")
    time_label = np.array(df_cls['day'])
    print("classes:", set(time_label))
        
    return full_matrix, time_label

In [ ]:
example_name = '887' # 
d_reds = [2, 4, 8, 16, 32, 64, 72] # [] or list of reduced dimensions
    
full_matrix, time_label = load_887_dataset()
visualize_data(full_matrix, time_label, colors=contrast_colors)
try:
    pca, projected_matrix = reduce_dimension(full_matrix, d_reds, example_name)
    data = {'full_matrix': full_matrix, 'time_label': time_label, 'pca': pca, 'projected_matrix': projected_matrix}
except:
    data = {'full_matrix': full_matrix, 'time_label': time_label}
save_files(data, example_name)